# HifazatTab — Phase 2 fallback: plain LoRA SFT

Use this ONLY if the GRPO notebook keeps OOMing. No rollouts, no generations,
no ref model — one forward/backward per batch. Fits any T4 easily.

Trains on the same 174 seed rows (tool JSON as the answer), pushes
`umeraamir69/hifazat-tab-sft-v1`. Then re-run Phase 1 for the AFTER number.

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
!pip install -q transformers accelerate bitsandbytes peft trl huggingface_hub datasets
import torch; print("cuda:", torch.cuda.is_available(), torch.cuda.device_count())

In [ ]:
from huggingface_hub import login
_tok = None
try:
    from google.colab import userdata
    _tok = userdata.get("HF_TOKEN")
except Exception:
    pass
if not _tok:
    try:
        from kaggle_secrets import UserSecretsClient
        _tok = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
if not _tok:
    from getpass import getpass
    _tok = getpass("Hugging Face token: ")
login(_tok)
del _tok


In [ ]:
import json, urllib.request
from datasets import Dataset
BASE = "https://raw.githubusercontent.com/umeraamir69/HifazatTab/main/data"
REFUSE = {
    "en": "I cannot help with that. I only answer price, stock, and loan-eligibility questions.",
    "hi": "\u092e\u0948\u0902\u0020\u0907\u0938\u092e\u0947\u0902\u0020\u092e\u0926\u0926\u0020\u0928\u0939\u0940\u0902\u0020\u0915\u0930\u0020\u0938\u0915\u0924\u093e\u0964\u0020\u092e\u0948\u0902\u0020\u0938\u093f\u0930\u094d\u092b\u0020\u0915\u0940\u092e\u0924\u002c\u0020\u0938\u094d\u091f\u0949\u0915\u0020\u0914\u0930\u0020\u0932\u094b\u0928\u002d\u092f\u094b\u0917\u094d\u092f\u0924\u093e\u0020\u092c\u0924\u093e\u0924\u093e\u0020\u0939\u0942\u0901\u0964",
    "ur": "\u0645\u06cc\u06ba\u0020\u0627\u0633\u0020\u0645\u06cc\u06ba\u0020\u0645\u062f\u062f\u0020\u0646\u06c1\u06cc\u06ba\u0020\u06a9\u0631\u0020\u0633\u06a9\u062a\u0627\u06d4\u0020\u0645\u06cc\u06ba\u0020\u0635\u0631\u0641\u0020\u0642\u06cc\u0645\u062a\u060c\u0020\u0627\u0633\u0679\u0627\u06a9\u0020\u0627\u0648\u0631\u0020\u0642\u0631\u0636\u0020\u06a9\u06cc\u0020\u0627\u06c1\u0644\u06cc\u062a\u0020\u0628\u062a\u0627\u062a\u0627\u0020\u06c1\u0648\u06ba\u06d4",
}
def load_seed():
    rows = []
    with urllib.request.urlopen(BASE + "/seed_tasks.jsonl") as f:
        for line in f:
            line = line.decode("utf-8").strip()
            if line: rows.append(json.loads(line))
    return rows
texts = []
for r in load_seed():
    if r.get("type") == "tool-task":
        tool, _, arg = r["tool"].partition(":")
        ans = '[{"tool_name": "%s", "parameters": {"arg": "%s"}}]' % (tool, arg)
    else:
        ans = REFUSE.get(r.get("lang", "en"), REFUSE["en"])
    texts.append("Question: " + r["prompt"] + "\nAnswer: " + ans)
ds = Dataset.from_dict({"text": texts})
print("sft rows:", len(ds))

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
MID = "CohereLabs/tiny-aya-fire"
tok = AutoTokenizer.from_pretrained(MID)
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(MID, load_in_4bit=True, device_map="auto", torch_dtype=torch.float16)
args = SFTConfig(output_dir="./hifazat-sft", num_train_epochs=3, per_device_train_batch_size=1, gradient_accumulation_steps=8, gradient_checkpointing=True, learning_rate=2e-4, logging_steps=10, save_steps=100, max_seq_length=256, packing=False, dataset_text_field="text", optim="paged_adamw_8bit", report_to="none")
trainer = SFTTrainer(model=model, args=args, train_dataset=ds, peft_config=LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05, task_type="CAUSAL_LM", target_modules=["q_proj", "v_proj"]))
trainer.train()
trainer.push_to_hub("umeraamir69/hifazat-tab-sft-v1")
print("pushed. Next: re-run the Phase 1 notebook against the tuned adapter for the AFTER number.")